# 04 — Feature Engineering
Build model-ready features for each coin individually. Rolling windows, z-scores, and cross-coin signals are computed per-coin so that no coin's data contaminates another's time series.

| # | Notebook | Reads → Writes |
|---|----------|----------------|
| 01 | Merge Raw Data | `raw/` → `merged/{coin}_5m_raw.parquet` |
| 02 | Clean & Label | `merged/` → `cleansed/{coin}_5m.parquet` |
| 03 | EDA | `cleansed/` |
| 03b | EDA — Downside Depegs | `cleansed/` |
| **▶ 04** | **Feature Engineering** | `cleansed/` → `features/{coin}_5m_features.parquet` |
| 05 | Build Pooled Dataset | `features/` → `features/pooled_5m.parquet` |
| 06 | Feature-Level EDA | `features/pooled_5m.parquet` |
| 07 | Feature Selection | `features/pooled_5m.parquet` → `features/selected_features.json` |
| 08 | Baseline Models | `features/pooled_5m.parquet` + `selected_features.json` → model results |
| 09 | Final Model (CatBoost) | + `selected_features.json` → `data/models/downside_depeg_catboost.cbm` + `downside_depeg_meta.json` |
| 10 | Threshold & Ops | `data/models/downside_depeg_meta.json` → threshold, alert metrics |
| 11 | LOEO Validation | `data/models/downside_depeg_meta.json` → leave-one-event-out results |
### Engineered feature groups added (68 new columns)
Pass-through columns from the cleansed file (raw OHLCV, on-chain raw, macro, labels) are retained alongside these.

| Group | Features | Description |
|-------|----------|-------------|
| Price deviation | 10 | Rolling mean/std/absmax at 15min/1h/4h |
| Price momentum | 7 | diff1, bars above 0.1%/0.3% thresholds at 15min/1h/4h |
| Intrabar volatility | 1 | (high−low)/close |
| On-chain flows | 17 | `total_net_flow_usd` sums at 1h/4h/1d; net-flow z-scores at 7d/30d; burn/mint z-scores at 7d/30d; mint/burn ratio at 1h/4h/1d |
| Curve DEX pressure | 5 | Rolling net sell at 15min/1h/4h, z-score, sell/buy ratio |
| Market returns | 8 | BTC/ETH returns (1h, 4h) & vol (4h, 1d), VIX diff, Fear&Greed diff |
| Cross-coin | ~6 | Other coins' `price_dev` reindexed to this coin's timestamps |
| Temporal | 4 | Hour, day-of-week, is_weekend, is_us_market_hours |
| Lags | 12 | price_dev, net_flow, Curve net_sell × lags 1/3/6/12 |
| **Total engineered** | **~76** | Exact count varies per coin (coin-specific on-chain features) |

### Target columns (all kept, all horizons)
All depeg label columns from the cleansed file are retained: `depeg`, `depeg_down`, and forward-looking targets at 5min/30min/1h/4h for both symmetric and downside variants.

In [1]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# ── Environment detection ─────────────────────────────────────────────────────
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Running in Google Colab — Drive mounted at /content/drive")
else:
    print("Running locally")

Running locally


In [2]:
DRIVE_PROJECT_PATH = "MyDrive/Capstone"   # ← Colab: path inside Google Drive
LOCAL_PROJECT_PATH = None                               # ← Local: set explicit path or None for auto-detect

if IN_COLAB:
    ROOT = Path("/content/drive") / DRIVE_PROJECT_PATH
elif LOCAL_PROJECT_PATH is not None:
    ROOT = Path(LOCAL_PROJECT_PATH)
else:
    _candidates = [Path.cwd()] + list(Path.cwd().parents)
    ROOT = next((p for p in _candidates if (p / "config" / "settings.py").exists()), None)
    if ROOT is None:
        raise FileNotFoundError(
            "Could not find project root (looked for config/settings.py). "
            "Set LOCAL_PROJECT_PATH above or run from within the project directory."
        )

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

CLEANSED_DIR = ROOT / "data" / "processed" / "cleansed"
OUT_DIR      = ROOT / "data" / "processed" / "features"   # ← change output location if needed

OUT_DIR.mkdir(parents=True, exist_ok=True)

from config.settings import STABLECOINS
COINS = list(STABLECOINS.keys())

print(f"Project root:  {ROOT}")
print(f"Input dir:     {CLEANSED_DIR}")
print(f"Output dir:    {OUT_DIR}")
print(f"Coins:         {COINS}")

Project root:  /Users/robertspringett/Education/CMU_MSBA/capstone_5min_global
Input dir:     /Users/robertspringett/Education/CMU_MSBA/capstone_5min_global/data/processed/cleansed
Output dir:    /Users/robertspringett/Education/CMU_MSBA/capstone_5min_global/data/processed/features
Coins:         ['usdt', 'usdc', 'dai', 'busd', 'ust', 'usde', 'rlusd']


## Feature Engineering Functions
All feature logic defined inline. Rolling windows operate within each coin's time series — this is why feature engineering runs per-coin before pooling.

In [3]:
# ── Time-bar constants ────────────────────────────────────────────────────────
BARS_PER_HOUR = 12
BARS_PER_4H   = 48
BARS_PER_DAY  = 288
BARS_PER_7D   = 2_016
BARS_PER_30D  = 8_640

# ── Curve column routing tables ───────────────────────────────────────────────
_COIN_CURVE_NET_SELL_COL = {
    "usdt":  "curve_3pool_usdt_net_sell_volume_usd",
    "usdc":  "curve_3pool_usdc_net_sell_volume_usd",
    "dai":   "curve_3pool_dai_net_sell_volume_usd",
    "busd":  "curve_busd_3crv_busd_net_sell_volume_usd",
    "ust":   None,
    "usde":  "curve_usde_usdc_usde_net_sell_volume_usd",
    "rlusd": "curve_rlusd_usdc_rlusd_net_sell_volume_usd",
}
_COIN_CURVE_SOLD_COL = {
    "usdt":  "curve_3pool_usdt_sold_volume_usd",
    "usdc":  "curve_3pool_usdc_sold_volume_usd",
    "dai":   "curve_3pool_dai_sold_volume_usd",
    "busd":  "curve_busd_3crv_busd_sold_volume_usd",
    "ust":   None,
    "usde":  "curve_usde_usdc_usde_sold_volume_usd",
    "rlusd": "curve_rlusd_usdc_rlusd_sold_volume_usd",
}
_COIN_CURVE_BOUGHT_COL = {
    "usdt":  "curve_3pool_usdt_bought_volume_usd",
    "usdc":  "curve_3pool_usdc_bought_volume_usd",
    "dai":   "curve_3pool_dai_bought_volume_usd",
    "busd":  "curve_busd_3crv_busd_bought_volume_usd",
    "ust":   None,
    "usde":  "curve_usde_usdc_usde_bought_volume_usd",
    "rlusd": "curve_rlusd_usdc_rlusd_bought_volume_usd",
}

# All depeg label columns carried through from the cleansed file
_LABEL_COLS = [
    "depeg", "depeg_down",
    "depeg_next_5min",     "depeg_next_5min_down",
    "depeg_next_30min",    "depeg_next_30min_down",
    "depeg_next_1h",       "depeg_next_1h_down",
    "depeg_next_4h",       "depeg_next_4h_down",
    "depeg_next_24h",      "depeg_next_24h_down",
]


def _col(df, name, default=0.0):
    """Safe column accessor — returns zeros if column is absent."""
    return df[name].fillna(default) if name in df.columns \
        else pd.Series(default, index=df.index, dtype="float64")


def _safe_zscore(series, window):
    """Rolling z-score; returns 0.0 where std is zero."""
    roll = series.rolling(window, min_periods=1)
    mu   = roll.mean()
    std  = roll.std(ddof=0)
    return (series - mu).div(std.replace(0.0, float("nan"))).fillna(0.0)


def add_price_features(df, coin_key):
    pd_ = _col(df, "price_dev")
    for w, tag in [(3, "15min"), (BARS_PER_HOUR, "1h"), (BARS_PER_4H, "4h")]:
        df[f"price_dev_mean_{tag}"]   = pd_.rolling(w, min_periods=1).mean()
        df[f"price_dev_std_{tag}"]    = pd_.rolling(w, min_periods=1).std(ddof=0)
        df[f"price_dev_absmax_{tag}"] = pd_.abs().rolling(w, min_periods=1).max()
    df["price_dev_diff1"] = pd_.diff(1).fillna(0.0)
    df["price_dev_acc"]   = df["price_dev_diff1"].diff(1).fillna(0.0)
    for w, tag in [(3, "15min"), (BARS_PER_HOUR, "1h"), (BARS_PER_4H, "4h")]:
        df[f"bars_above_01pct_{tag}"] = (pd_.abs() > 0.001).astype(float).rolling(w, min_periods=1).sum()
        df[f"bars_above_03pct_{tag}"] = (pd_.abs() > 0.003).astype(float).rolling(w, min_periods=1).sum()
    hi = _col(df, "coinapi_high")
    lo = _col(df, "coinapi_low")
    cl = _col(df, "coinapi_close", default=1.0).replace(0.0, float("nan")).fillna(1.0)
    df["intrabar_range"] = (hi - lo) / cl
    return df


def add_onchain_features(df, coin_key):
    # Compute unified net flow
    # USDT: ETH treasury + TRON treasury + Omni (Bitcoin) treasury
    # All others: on-chain mint/burn net flow
    if coin_key == "usdt":
        net = (
            _col(df, "treasury_net_flow_usd")
            + _col(df, "tron_treasury_net_flow_usd")
            + _col(df, "omni_treasury_net_flow_usd")
        )
    else:
        net = _col(df, "net_flow_usd")
    df["total_net_flow_usd"] = net

    for w, tag in [(BARS_PER_HOUR, "1h"), (BARS_PER_4H, "4h"), (BARS_PER_DAY, "1d")]:
        df[f"net_flow_sum_{tag}"] = net.rolling(w, min_periods=1).sum()
    df["net_flow_zscore_1d"]  = _safe_zscore(net, BARS_PER_DAY)    # ← new 1d variant
    df["net_flow_zscore_7d"]  = _safe_zscore(net, BARS_PER_7D)
    df["net_flow_zscore_30d"] = _safe_zscore(net, BARS_PER_30D)

    mint = _col(df, "mint_volume_usd")
    burn = _col(df, "burn_volume_usd")
    for w, tag in [(BARS_PER_HOUR, "1h"), (BARS_PER_4H, "4h")]:
        df[f"mint_sum_{tag}"] = mint.rolling(w, min_periods=1).sum()
        df[f"burn_sum_{tag}"] = burn.rolling(w, min_periods=1).sum()
    df["burn_zscore_1d"]  = _safe_zscore(burn, BARS_PER_DAY)       # ← new 1d variant
    df["burn_zscore_7d"]  = _safe_zscore(burn, BARS_PER_7D)
    df["burn_zscore_30d"] = _safe_zscore(burn, BARS_PER_30D)
    df["mint_zscore_1d"]  = _safe_zscore(mint, BARS_PER_DAY)       # ← new 1d variant
    df["mint_zscore_7d"]  = _safe_zscore(mint, BARS_PER_7D)
    df["mint_zscore_30d"] = _safe_zscore(mint, BARS_PER_30D)

    mint_1h = mint.rolling(BARS_PER_HOUR, min_periods=1).sum()
    mint_4h = mint.rolling(BARS_PER_4H,   min_periods=1).sum()
    mint_1d = mint.rolling(BARS_PER_DAY,  min_periods=1).sum()
    burn_1h = burn.rolling(BARS_PER_HOUR, min_periods=1).sum()
    burn_4h = burn.rolling(BARS_PER_4H,   min_periods=1).sum()
    burn_1d = burn.rolling(BARS_PER_DAY,  min_periods=1).sum()
    df["mint_burn_ratio_1h"] = mint_1h / (burn_1h + 1.0)
    df["mint_burn_ratio_4h"] = mint_4h / (burn_4h + 1.0)
    df["mint_burn_ratio_1d"] = mint_1d / (burn_1d + 1.0)

    # ── Rate-of-change / acceleration ──────────────────────────────────────────
    df["burn_acc_1h"]     = (burn_1h - burn_1h.shift(BARS_PER_HOUR)).fillna(0.0)
    df["mint_acc_1h"]     = (mint_1h - mint_1h.shift(BARS_PER_HOUR)).fillna(0.0)
    df["burn_rate_ratio"] = burn_1h / (burn_4h / 4 + 1.0)
    df["mint_rate_ratio"] = mint_1h / (mint_4h / 4 + 1.0)
    net_1h = net.rolling(BARS_PER_HOUR, min_periods=1).sum()
    df["net_flow_acc_1h"]         = (net_1h - net_1h.shift(BARS_PER_HOUR)).fillna(0.0)
    df["burn_acc_zscore_1d"]      = _safe_zscore(df["burn_acc_1h"],     BARS_PER_DAY)    # ← new 1d RoC
    df["burn_acc_zscore_7d"]      = _safe_zscore(df["burn_acc_1h"],     BARS_PER_7D)
    df["burn_acc_zscore_30d"]     = _safe_zscore(df["burn_acc_1h"],     BARS_PER_30D)
    df["net_flow_acc_zscore_1d"]  = _safe_zscore(df["net_flow_acc_1h"], BARS_PER_DAY)    # ← new 1d RoC
    df["net_flow_acc_zscore_7d"]  = _safe_zscore(df["net_flow_acc_1h"], BARS_PER_7D)
    df["net_flow_acc_zscore_30d"] = _safe_zscore(df["net_flow_acc_1h"], BARS_PER_30D)

    for w, tag in [(BARS_PER_HOUR, "1h"), (BARS_PER_4H, "4h")]:
        df[f"net_flow_vol_{tag}"]  = net.rolling(w, min_periods=2).std(ddof=0).fillna(0.0)
        df[f"burn_vol_{tag}"]      = burn.rolling(w, min_periods=2).std(ddof=0).fillna(0.0)

    _net_vol_4h  = net.rolling(BARS_PER_4H,  min_periods=2).std(ddof=0).fillna(0.0)
    _burn_vol_4h = burn.rolling(BARS_PER_4H, min_periods=2).std(ddof=0).fillna(0.0)
    df["net_flow_vol_zscore_1d"]  = _safe_zscore(_net_vol_4h,  BARS_PER_DAY)    # ← new 1d variant
    df["net_flow_vol_zscore_7d"]  = _safe_zscore(_net_vol_4h,  BARS_PER_7D)
    df["net_flow_vol_zscore_30d"] = _safe_zscore(_net_vol_4h,  BARS_PER_30D)
    df["burn_vol_zscore_1d"]      = _safe_zscore(_burn_vol_4h, BARS_PER_DAY)    # ← new 1d variant
    df["burn_vol_zscore_7d"]      = _safe_zscore(_burn_vol_4h, BARS_PER_7D)
    df["burn_vol_zscore_30d"]     = _safe_zscore(_burn_vol_4h, BARS_PER_30D)

    if coin_key == "usdt":
        for src, prefix in [("eth", "treasury"), ("tron", "tron_treasury")]:
            for w, tag in [(BARS_PER_HOUR, "1h"), (BARS_PER_4H, "4h")]:
                df[f"{src}_treasury_inflow_sum_{tag}"]  = _col(df, f"{prefix}_inflow_volume_usd").rolling(w, min_periods=1).sum()
                df[f"{src}_treasury_outflow_sum_{tag}"] = _col(df, f"{prefix}_outflow_volume_usd").rolling(w, min_periods=1).sum()
        for w, tag in [(BARS_PER_HOUR, "1h"), (BARS_PER_4H, "4h")]:
            df[f"omni_treasury_inflow_sum_{tag}"]  = _col(df, "omni_treasury_inflow_volume_usd").rolling(w, min_periods=1).sum()
            df[f"omni_treasury_outflow_sum_{tag}"] = _col(df, "omni_treasury_outflow_volume_usd").rolling(w, min_periods=1).sum()
    if coin_key == "usdc":
        sol_net = _col(df, "sol_net_flow_usd")
        for w, tag in [(BARS_PER_HOUR, "1h"), (BARS_PER_4H, "4h")]:
            df[f"sol_net_flow_sum_{tag}"] = sol_net.rolling(w, min_periods=1).sum()
        df["sol_net_flow_zscore_30d"] = _safe_zscore(sol_net, BARS_PER_30D)
        df["sol_net_flow_zscore_7d"]  = _safe_zscore(sol_net, BARS_PER_7D)
    if coin_key == "rlusd":
        xrpl_net = _col(df, "xrpl_net_flow_usd")
        for w, tag in [(BARS_PER_HOUR, "1h"), (BARS_PER_4H, "4h")]:
            df[f"xrpl_net_flow_sum_{tag}"] = xrpl_net.rolling(w, min_periods=1).sum()
    return df


def add_curve_features(df, coin_key):
    if coin_key == "ust":
        net_sell = _col(df, "curve_ust_3crv_ust_net_sell_volume_usd") + _col(df, "curve_ust_wormhole_3crv_wust_net_sell_volume_usd")
        sold     = _col(df, "curve_ust_3crv_ust_sold_volume_usd")     + _col(df, "curve_ust_wormhole_3crv_wust_sold_volume_usd")
        bought   = _col(df, "curve_ust_3crv_ust_bought_volume_usd")   + _col(df, "curve_ust_wormhole_3crv_wust_bought_volume_usd")
    else:
        net_sell = _col(df, _COIN_CURVE_NET_SELL_COL.get(coin_key, ""))
        sold     = _col(df, _COIN_CURVE_SOLD_COL.get(coin_key, ""))
        bought   = _col(df, _COIN_CURVE_BOUGHT_COL.get(coin_key, ""))
    for w, tag in [(3, "15min"), (BARS_PER_HOUR, "1h"), (BARS_PER_4H, "4h")]:
        df[f"curve_net_sell_sum_{tag}"] = net_sell.rolling(w, min_periods=1).sum()
    df["curve_net_sell_zscore_1d"]  = _safe_zscore(net_sell, BARS_PER_DAY)    # ← new 1d variant
    df["curve_net_sell_zscore_7d"]  = _safe_zscore(net_sell, BARS_PER_7D)
    df["curve_net_sell_zscore_30d"] = _safe_zscore(net_sell, BARS_PER_30D)

    # ── Curve rate-of-change ──────────────────────────────────────────────────
    curve_1h = net_sell.rolling(BARS_PER_HOUR, min_periods=1).sum()
    curve_4h = net_sell.rolling(BARS_PER_4H,   min_periods=1).sum()
    df["curve_acc_1h"]          = (curve_1h - curve_1h.shift(BARS_PER_HOUR)).fillna(0.0)
    df["curve_rate_ratio"]      = curve_1h / (curve_4h / 4 + 1.0)
    df["curve_acc_zscore_1d"]   = _safe_zscore(df["curve_acc_1h"], BARS_PER_DAY)    # ← new 1d RoC
    df["curve_acc_zscore_7d"]   = _safe_zscore(df["curve_acc_1h"], BARS_PER_7D)
    df["curve_acc_zscore_30d"]  = _safe_zscore(df["curve_acc_1h"], BARS_PER_30D)

    df["curve_sell_buy_ratio_1h"] = sold.rolling(BARS_PER_HOUR, min_periods=1).sum() / \
                                    (bought.rolling(BARS_PER_HOUR, min_periods=1).sum() + 1.0)
    for w, tag in [(BARS_PER_HOUR, "1h"), (BARS_PER_4H, "4h")]:
        df[f"curve_net_sell_vol_{tag}"] = net_sell.rolling(w, min_periods=2).std(ddof=0).fillna(0.0)

    _curve_vol_4h = net_sell.rolling(BARS_PER_4H, min_periods=2).std(ddof=0).fillna(0.0)
    df["curve_net_sell_vol_zscore_1d"]  = _safe_zscore(_curve_vol_4h, BARS_PER_DAY)    # ← new 1d variant
    df["curve_net_sell_vol_zscore_7d"]  = _safe_zscore(_curve_vol_4h, BARS_PER_7D)
    df["curve_net_sell_vol_zscore_30d"] = _safe_zscore(_curve_vol_4h, BARS_PER_30D)

    # ── Cross-pool aggregate: total 3pool sell volume ─────────────────────────
    curve_3pool_total_sell = (
        _col(df, "curve_3pool_usdt_sold_volume_usd")
        + _col(df, "curve_3pool_usdc_sold_volume_usd")
        + _col(df, "curve_3pool_dai_sold_volume_usd")
    )
    for w, tag in [(BARS_PER_HOUR, "1h"), (BARS_PER_4H, "4h")]:
        df[f"curve_3pool_total_sell_sum_{tag}"] = curve_3pool_total_sell.rolling(w, min_periods=1).sum()
    df["curve_3pool_total_sell_zscore_30d"] = _safe_zscore(curve_3pool_total_sell, BARS_PER_30D)

    return df


def add_market_features(df):
    btc = _col(df, "binance_btc_close", default=float("nan")).ffill()
    eth = _col(df, "binance_eth_close", default=float("nan")).ffill()
    vix = _col(df, "vix", default=float("nan")).ffill()
    fg  = _col(df, "fear_greed", default=float("nan")).ffill()

    df["btc_return_1h"] = btc.pct_change(BARS_PER_HOUR).fillna(0.0)
    df["btc_return_4h"] = btc.pct_change(BARS_PER_4H).fillna(0.0)
    df["eth_return_1h"] = eth.pct_change(BARS_PER_HOUR).fillna(0.0)
    df["eth_return_4h"] = eth.pct_change(BARS_PER_4H).fillna(0.0)
    df["btc_return_24h"] = btc.pct_change(BARS_PER_DAY).fillna(0.0)
    df["eth_return_24h"] = eth.pct_change(BARS_PER_DAY).fillna(0.0)
    df["btc_drawdown_accel"] = (df["btc_return_1h"] - df["btc_return_4h"]).fillna(0.0)
    df["eth_drawdown_accel"] = (df["eth_return_1h"] - df["eth_return_4h"]).fillna(0.0)

    df["btc_return_1h_zscore_30d"] = _safe_zscore(df["btc_return_1h"], BARS_PER_30D)
    df["btc_return_4h_zscore_30d"] = _safe_zscore(df["btc_return_4h"], BARS_PER_30D)
    df["eth_return_1h_zscore_30d"] = _safe_zscore(df["eth_return_1h"], BARS_PER_30D)
    df["eth_return_4h_zscore_30d"] = _safe_zscore(df["eth_return_4h"], BARS_PER_30D)

    btc_ret1 = btc.pct_change(1).fillna(0.0)
    df["btc_vol_4h"] = btc_ret1.rolling(BARS_PER_4H,  min_periods=1).std(ddof=0)
    df["btc_vol_1d"] = btc_ret1.rolling(BARS_PER_DAY, min_periods=1).std(ddof=0)

    df["vix_diff_1d"] = vix.diff(BARS_PER_DAY).fillna(0.0)
    df["vix_diff_1d_zscore_30d"] = _safe_zscore(df["vix_diff_1d"], BARS_PER_30D)

    fg_filled = fg.fillna(50.0)
    df["fear_greed_level"]      = fg_filled
    df["fear_greed_zscore_30d"] = _safe_zscore(fg_filled, BARS_PER_30D)
    df["fear_greed_diff_1d"]    = fg.diff(BARS_PER_DAY).fillna(0.0)
    df["fear_greed_diff_7d"]    = fg.diff(BARS_PER_7D).fillna(0.0)

    return df


def add_cross_coin_features(df, coin_key, price_dev_map):
    for other, series in price_dev_map.items():
        if other == coin_key:
            continue
        df[f"cross_{other}_price_dev"] = series.reindex(df.index).fillna(0.0).values

    for proxy in ["usdt", "usdc"]:
        col = f"cross_{proxy}_price_dev"
        if col not in df.columns:
            df[col] = 0.0
    return df


def add_temporal_features(df):
    idx = df.index
    utc_idx = idx.tz_localize("UTC") if (not hasattr(idx, "tz") or idx.tz is None) else idx
    df["hour_of_day"]      = utc_idx.hour
    df["day_of_week"]      = utc_idx.dayofweek
    df["is_weekend"]       = (utc_idx.dayofweek >= 5).astype("int8")
    us_market = (utc_idx.dayofweek < 5) & \
                (utc_idx.hour * 60 + utc_idx.minute >= 13 * 60 + 30) & \
                (utc_idx.hour * 60 + utc_idx.minute < 20 * 60)
    df["is_us_market_hours"] = us_market.astype("int8")
    return df


def add_lag_features(df, coin_key):
    pd_  = _col(df, "price_dev")
    nf_  = _col(df, "total_net_flow_usd")
    cv_  = (_col(df, "curve_ust_3crv_ust_net_sell_volume_usd") + _col(df, "curve_ust_wormhole_3crv_wust_net_sell_volume_usd")) \
           if coin_key == "ust" else _col(df, _COIN_CURVE_NET_SELL_COL.get(coin_key, ""))
    for lag in [1, 3, 6, 12]:
        df[f"lag{lag}_price_dev"]      = pd_.shift(lag).fillna(0.0)
        df[f"lag{lag}_net_flow_usd"]   = nf_.shift(lag).fillna(0.0)
        df[f"lag{lag}_curve_net_sell"] = cv_.shift(lag).fillna(0.0)
    return df


def engineer_coin(coin_key, price_dev_map):
    df = pd.read_parquet(CLEANSED_DIR / f"{coin_key}_5m.parquet")

    for col in _LABEL_COLS:
        if col in df.columns and pd.api.types.is_extension_array_dtype(df[col]):
            df[col] = df[col].astype("float64")

    df = add_price_features(df, coin_key)
    df = add_onchain_features(df, coin_key)
    df = add_curve_features(df, coin_key)
    df = add_market_features(df)
    df = add_cross_coin_features(df, coin_key, price_dev_map)
    df = add_temporal_features(df)
    df = add_lag_features(df, coin_key)
    return df


print("Feature functions defined.")
print(f"  1h={BARS_PER_HOUR} bars | 4h={BARS_PER_4H} | 1d={BARS_PER_DAY} | 7d={BARS_PER_7D} | 30d={BARS_PER_30D}")
print(f"  Z-score windows: 1d, 7d, 30d for net_flow, burn, mint, vol, acc; 1d, 7d, 30d for curve")
print(f"  Label columns retained: {_LABEL_COLS}")

Feature functions defined.
  1h=12 bars | 4h=48 | 1d=288 | 7d=2016 | 30d=8640
  Z-score windows: 1d, 7d, 30d for net_flow, burn, mint, vol, acc; 1d, 7d, 30d for curve
  Label columns retained: ['depeg', 'depeg_down', 'depeg_next_5min', 'depeg_next_5min_down', 'depeg_next_30min', 'depeg_next_30min_down', 'depeg_next_1h', 'depeg_next_1h_down', 'depeg_next_4h', 'depeg_next_4h_down', 'depeg_next_24h', 'depeg_next_24h_down']


## 1. Pre-load price_dev for Cross-Coin Features
Each coin's `price_dev` is loaded upfront so cross-coin features can be aligned by timestamp.

In [4]:
price_dev_map = {}
for coin_key in COINS:
    path = CLEANSED_DIR / f"{coin_key}_5m.parquet"
    if not path.exists():
        print(f"  [warn] {path.name} not found")
        continue
    df_tmp = pd.read_parquet(path, columns=["price_dev"])
    price_dev_map[coin_key] = df_tmp["price_dev"].astype("float64")
    print(f"  {coin_key.upper():6s}  {len(df_tmp):>9,} rows  "
          f"price_dev range: [{df_tmp['price_dev'].min():.4f}, {df_tmp['price_dev'].max():.4f}]")

print(f"\nLoaded price_dev for {len(price_dev_map)} coins.")

  USDT      897,936 rows  price_dev range: [-0.1382, 0.0777]
  USDC      775,506 rows  price_dev range: [-0.4952, 0.7072]
  DAI       828,963 rows  price_dev range: [-0.4990, 0.5403]
  BUSD      370,848 rows  price_dev range: [-0.0070, 0.0551]
  UST       153,961 rows  price_dev range: [-0.7494, 0.0490]
  USDE      200,928 rows  price_dev range: [-0.0187, 0.0087]
  RLUSD      96,192 rows  price_dev range: [-0.0088, 0.0026]

Loaded price_dev for 7 coins.


## 2. Engineer Features — All Coins
Each coin is processed independently. Rolling windows stay within each coin's time series.

In [5]:
for coin_key in COINS:
    src_path = CLEANSED_DIR / f"{coin_key}_5m.parquet"
    if not src_path.exists():
        print(f"[{coin_key.upper()}] ✗  {src_path.name} not found — skipping")
        continue

    print(f"[{coin_key.upper()}] Engineering features…", end="", flush=True)
    df = engineer_coin(coin_key, price_dev_map)

    out_path = OUT_DIR / f"{coin_key}_5m_features.parquet"
    df.to_parquet(out_path)

    depeg_rate = df["depeg_next_1h"].mean() if "depeg_next_1h" in df.columns else float("nan")
    print(f"  {len(df):>9,} rows × {len(df.columns)} cols  "
          f"depeg_next_1h={depeg_rate:.2%}  → {out_path.name}")

print(f"\n✓  Saved per-coin feature files to {OUT_DIR}")

[USDT] Engineering features…

    897,936 rows × 211 cols  depeg_next_1h=9.92%  → usdt_5m_features.parquet
[USDC] Engineering features…

    775,506 rows × 215 cols  depeg_next_1h=5.93%  → usdc_5m_features.parquet
[DAI] Engineering features…

    828,963 rows × 172 cols  depeg_next_1h=11.89%  → dai_5m_features.parquet
[BUSD] Engineering features…

    370,848 rows × 167 cols  depeg_next_1h=0.38%  → busd_5m_features.parquet
[UST] Engineering features…

    153,961 rows × 172 cols  depeg_next_1h=6.19%  → ust_5m_features.parquet
[USDE] Engineering features…

    200,928 rows × 167 cols  depeg_next_1h=0.10%  → usde_5m_features.parquet
[RLUSD] Engineering features…

     96,192 rows × 174 cols  depeg_next_1h=0.00%  → rlusd_5m_features.parquet

✓  Saved per-coin feature files to /Users/robertspringett/Education/CMU_MSBA/capstone_5min_global/data/processed/features


## 3. Output Confirmation

In [6]:
print(f"Output: {OUT_DIR}\n")
for coin_key in COINS:
    p = OUT_DIR / f"{coin_key}_5m_features.parquet"
    if p.exists():
        size_mb = p.stat().st_size / 1e6
        print(f"  ✓  {p.name:<35s}  {size_mb:6.1f} MB")
    else:
        print(f"  ✗  {p.name}  — not found")

Output: /Users/robertspringett/Education/CMU_MSBA/capstone_5min_global/data/processed/features

  ✓  usdt_5m_features.parquet              453.3 MB
  ✓  usdc_5m_features.parquet              723.8 MB
  ✓  dai_5m_features.parquet               602.2 MB
  ✓  busd_5m_features.parquet              151.2 MB
  ✓  ust_5m_features.parquet                64.4 MB
  ✓  usde_5m_features.parquet              136.0 MB
  ✓  rlusd_5m_features.parquet              49.2 MB


## 4. Next Step
Run `05_build_pooled_dataset.ipynb` to select the 76 common columns and stack all coins into a single modeling-ready dataset.